<a href="https://colab.research.google.com/github/thuviettran/demo-github/blob/main/movie_v3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Hybrid Movie Recommender tối ưu cho MovieLens 1M
# Bao gồm: re-index ID, chuẩn hóa rating, genre embedding, early stopping

import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from tensorflow.keras.layers import (
    Input, Embedding, Flatten, Dense, Concatenate,
    Dropout, GlobalAveragePooling1D
)
from tensorflow.keras.models import Model
from tensorflow.keras.callbacks import EarlyStopping

# =====================
# 1. Load dữ liệu .dat
# =====================
ratings = pd.read_csv(
    "ratings.dat",
    sep="::",
    engine="python",
    names=["userId", "movieId", "rating", "timestamp"]
)

movies = pd.read_csv(
    "movies.dat",
    sep="::",
    engine="python",
    names=["movieId", "title", "genres"],
    encoding="latin-1"
)

# =====================
# 2. Re-index ID (quan trọng)
# Fit encoder trên TẬP HỢP movieId của cả ratings + movies để tránh unseen labels
# =====================
user_encoder = LabelEncoder()
movie_encoder = LabelEncoder()

ratings["userId"] = user_encoder.fit_transform(ratings["userId"])

# Fit movie encoder trên union movieId
all_movie_ids = pd.concat([ratings["movieId"], movies["movieId"]]).unique()
movie_encoder.fit(all_movie_ids)

ratings["movieId"] = movie_encoder.transform(ratings["movieId"])
movies["movieId"] = movie_encoder.transform(movies["movieId"])

num_users = ratings["userId"].nunique()
num_movies = len(all_movie_ids)

# =====================
# 3. Xử lý genre → danh sách index
# =====================
movies["genre_list"] = movies["genres"].apply(lambda x: x.split("|"))

all_genres = sorted({g for sub in movies["genre_list"] for g in sub})
genre_to_idx = {g: i + 1 for i, g in enumerate(all_genres)}  # 0 = padding

max_genres = movies["genre_list"].apply(len).max()
num_genres = len(all_genres) + 1


def encode_genres(genre_list):
    idxs = [genre_to_idx[g] for g in genre_list]
    return idxs + [0] * (max_genres - len(idxs))


movies["genre_encoded"] = movies["genre_list"].apply(encode_genres)

# =====================
# 4. Merge dữ liệu
# =====================
df = ratings.merge(movies[["movieId", "genre_encoded"]], on="movieId")

# =====================
# 5. Chuẩn hóa rating
# =====================
y = df["rating"].values.astype("float32")
y_mean = y.mean()
y_std = y.std()
y = (y - y_mean) / y_std

user_input = df["userId"].values
movie_input = df["movieId"].values
genre_input = np.stack(df["genre_encoded"].values)

# =====================
# 6. Train/Test split
# =====================
(
    user_train, user_test,
    movie_train, movie_test,
    genre_train, genre_test,
    y_train, y_test
) = train_test_split(
    user_input, movie_input, genre_input, y,
    test_size=0.2,
    random_state=42
)

# =====================
# 7. Xây dựng Hybrid Model tối ưu
# =====================
embedding_dim = 64

user_in = Input(shape=(1,))
movie_in = Input(shape=(1,))
genre_in = Input(shape=(max_genres,))

# User & Movie embedding (Collaborative Filtering)
user_emb = Embedding(num_users, embedding_dim)(user_in)
movie_emb = Embedding(num_movies, embedding_dim)(movie_in)

user_vec = Flatten()(user_emb)
movie_vec = Flatten()(movie_emb)

# Genre embedding (Content-based)
genre_emb = Embedding(num_genres, 32, mask_zero=True)(genre_in)
genre_vec = GlobalAveragePooling1D()(genre_emb)

# Combine tất cả
x = Concatenate()([user_vec, movie_vec, genre_vec])

# Deep layers
x = Dense(256, activation="relu")(x)
x = Dropout(0.4)(x)
x = Dense(128, activation="relu")(x)
x = Dense(64, activation="relu")(x)

out = Dense(1, activation="linear")(x)

model = Model(inputs=[user_in, movie_in, genre_in], outputs=out)

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss="mse",
    metrics=["mae"]
)

model.summary()

# =====================
# 8. Train với EarlyStopping
# =====================
early_stop = EarlyStopping(
    monitor="val_loss",
    patience=3,
    restore_best_weights=True
)

history = model.fit(
    [user_train, movie_train, genre_train],
    y_train,
    validation_split=0.1,
    epochs=20,
    batch_size=512,
    callbacks=[early_stop],
    verbose=1
)

# =====================
# 9. Evaluate
# =====================
loss, mae = model.evaluate(
    [user_test, movie_test, genre_test],
    y_test,
    verbose=0
)

rmse = np.sqrt(loss)

print(f"Test RMSE: {rmse:.4f}")
print(f"Test MAE: {mae:.4f}")

# =====================
# 10. Hàm gợi ý phim
# =====================
movies_full = movies.copy()


def recommend(user_id_raw, top_k=10):
    user_id = user_encoder.transform([user_id_raw])[0]

    user_arr = np.full(len(movies_full), user_id)
    movie_arr = movies_full["movieId"].values
    genre_arr = np.stack(movies_full["genre_encoded"].values)

    preds = model.predict([user_arr, movie_arr, genre_arr], verbose=0).flatten()

    # scale ngược rating
    preds = preds * y_std + y_mean

    movies_full["pred_rating"] = preds

    return movies_full.sort_values("pred_rating", ascending=False).head(top_k)[
        ["title", "genres", "pred_rating"]
    ]


# Ví dụ gợi ý cho user gốc = 1
print(recommend(1))


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 1)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ input_layer_1       │ (None, 1)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ input_layer_2       │ (None, 6)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding           │ (None, 1, 64)     │    386,560 │ input_layer[0][0] │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_1         │ (None, 1, 64)     │    248,512 │ input_layer_1[0]… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_2         │ (None, 6, 32)     │        608 │ input_layer_2[0]… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ not_equal           │ (None, 6)         │          0 │ input_layer_2[0]… │
│ (NotEqual)          │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ flatten (Flatten)   │ (None, 64)        │          0 │ embedding[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ flatten_1 (Flatten) │ (None, 64)        │          0 │ embedding_1[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ global_average_poo… │ (None, 32)        │          0 │ embedding_2[0][0… │
│ (GlobalAveragePool… │                   │            │ not_equal[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate         │ (None, 160)       │          0 │ flatten[0][0],    │
│ (Concatenate)       │                   │            │ flatten_1[0][0],  │
│                     │                   │            │ global_average_p… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense (Dense)       │ (None, 256)       │     41,216 │ concatenate[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout (Dropout)   │ (None, 256)       │          0 │ dense[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_1 (Dense)     │ (None, 128)       │     32,896 │ dropout[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_2 (Dense)     │ (None, 64)        │      8,256 │ dense_1[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_3 (Dense)     │ (None, 1)         │         65 │ dense_2[0][0]     │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 718,113 (2.74 MB)

 Trainable params: 718,113 (2.74 MB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/20
1407/1407 ━━━━━━━━━━━━━━━━━━━━ 11s 5ms/step - loss: 0.7427 - mae: 0.6849 - val_loss: 0.6508 - val_mae: 0.6358
Epoch 2/20
1407/1407 ━━━━━━━━━━━━━━━━━━━━ 4s 3ms/step - loss: 0.6349 - mae: 0.6278 - val_loss: 0.6267 - val_mae: 0.6255
Epoch 3/20
1407/1407 ━━━━━━━━━━━━━━━━━━━━ 5s 3ms/step - loss: 0.6027 - mae: 0.6122 - val_loss: 0.6108 - val_mae: 0.6171
Epoch 4/20
1407/1407 ━━━━━━━━━━━━━━━━━━━━ 4s 3ms/step - loss: 0.5763 - mae: 0.5990 - val_loss: 0.6027 - val_mae: 0.6119
Epoch 5/20
1407/1407 ━━━━━━━━━━━━━━━━━━━━ 4s 3ms/step - loss: 0.5543 - mae: 0.5870 - val_loss: 0.6010 - val_mae: 0.6118
Epoch 6/20
1407/1407 ━━━━━━━━━━━━━━━━━━━━ 5s 3ms/step - loss: 0.5349 - mae: 0.5762 - val_loss: 0.5961 - val_mae: 0.6082
Epoch 7/20
1407/1407 ━━━━━━━━━━━━━━━━━━━━ 4s 3ms/step - loss: 0.5188 - mae: 0.5675 - val_loss: 0.5969 - val_mae: 0.6088
Epoch 8/20
1407/1407 ━━━━━━━━━━━━━━━━━━━━ 4s 3ms/step - loss: 0.5021 - mae: 0.5577 - val_loss: 0.5976 - val_mae: 0.6070
Epoch 9/20
1407/1407 ━━━━━━━━━━━━━━━━━━